In [3]:
from pathlib import Path
from copy import deepcopy
import os
import time
import traceback
from typing import Optional

import pandas as pd

os.chdir("/mnt/primary/Resource-Prediction")
os.environ["GRB_LICENSE_FILE"] = str(Path("./gurobi/gurobi.lic").resolve())

from carbon_scheduling.src.carbon import CarbonProfile
from carbon_scheduling.src.plot import plot_ci_and_schedule_aligned_static
from carbon_scheduling.src.synthetic import synthesize_df_test
from carbon_scheduling.src.optimise import OptimiserMILP
from carbon_scheduling.src.scheduler import UncertaintyScheduler
from carbon_scheduling.src.structs import Logger
from carbon_scheduling.src.loading import df_to_queryspecs_pred_true
from carbon_scheduling.src.penalty import *
from carbon_scheduling.src.objective import ObjectiveFunction
from carbon_scheduling.src.filter import *
from carbon_scheduling.src.utilities import *
from carbon_scheduling.config import *


def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def _safe_int(x):
    try:
        return int(x)
    except Exception:
        return None


def run_single_scheduler_trial(
    *,
    csv_path: str,
    start: str,
    duration_hours: int,
    n_queries: int,
    label: str,
    penalties: list,
    repeat_idx: int,
    seed: int,
    reopt_threshold: float = 0.5,
    verbose_logger: bool = False,
    verbose_filter: bool = False,
) -> dict:
    """
    Run one scheduler trial safely.

    Returns a flat dict suitable for a DataFrame row.
    Never raises; failures are captured in the returned row.
    """
    cp = None
    specs_true = None
    res = None
    fstats = None

    try:
        cp = CarbonProfile.from_csv(
            csv_path=csv_path,
            use_lifecycle=True,
            start=start,
            duration_hours=duration_hours,
            upsample_to_sec=1,
        )
        
        bound_penalties = []
        for p in penalties:
            p_local = deepcopy(p)
            if hasattr(p_local, "bind"):
                p_local = p_local.bind(cp)
            bound_penalties.append(p_local)

        df_test = synthesize_df_test(
            n_queries=n_queries,
            n_runs=N_RUNS,
            base_median=BASE_MEDIAN,
            max_median=MAX_MEDIAN,
            cpu_min=CPU_MIN,
            cpu_max=CPU_MAX,
            ram_min=RAM_MIN,
            ram_max=RAM_MAX,
            priority_high=PRIORITY_HIGH,
            priority_low=PRIORITY_LOW,
            priority_w0=PRIORITY_W0,
            priority_w1=PRIORITY_w1,
            seed=seed,
        )

        specs_true = df_to_queryspecs_pred_true(
            df_test,
            slot_sec=cp.slot_sec,
            cpu_cap=MAX_CPU,
            ram_cap=MAX_RAM,
        )

        logger = Logger(verbose_logger)

        #query_filter_pipeline = QueryFilterPipeline(
        #    low_carbon_filter=LowCarbonFilter(
        #        q_lo=0.50,
        #        q_hi=0.20,
        #        cov_lo=0.05,
        #        cov_hi=0.30,
        #        keep_first=False,
        #    ),
        #    verbose=verbose_filter,
        #)
        query_filter_pipeline = None

        optimiser = OptimiserMILP(
            cpu_max=MAX_CPU,
            ram_max=MAX_RAM,
            query_filter_pipeline=query_filter_pipeline,
            objective=ObjectiveFunction(),
            penalties=bound_penalties,
            step_slots=TIME_SLOT_GRANULARITY,
            horizon_slots=HORIZON_TIME,
            mip_gap=None,
            time_limit_s=300,
            threads=8,
            logger=logger,
            verbose=False,
            seed=seed,
        )

        scheduler = UncertaintyScheduler(
            name=f"{label}_{duration_hours}h_q{n_queries}_r{repeat_idx}",
            label=f"{label}_{duration_hours}h_q{n_queries}",
            optimiser=optimiser,
            cp=cp,
            specs=specs_true,
            oracle_planning=False,
            use_true_for_execution=True,
            logger=logger,
            reopt_threshold=reopt_threshold,
            extra={
                "duration_hours": duration_hours,
                "n_queries": n_queries,
                "repeat_idx": repeat_idx,
                "seed": seed,
            },
        )

        res = scheduler.run()
        fstats = getattr(query_filter_pipeline, "last_stats", None) if query_filter_pipeline is not None else None

        res_summary = {f"{k}": v for k, v in res.summary().items()} if res is not None else {}
        
        return {
            "label": label,
            "duration_hours": duration_hours,
            "n_queries": n_queries,
            "repeat_idx": repeat_idx,
            "seed": seed,
            "success": True,
            "error_type": None,
            "error_message": None,
            "ci_cov": _safe_float(cp.ci_cov()) if cp is not None else None,
            "num_slots": _safe_int(cp.num_slots) if cp is not None else None,
            "num_queries_actual": _safe_int(len(specs_true)) if specs_true is not None else None,
            "raw_candidates": _safe_int(fstats.raw_total) if fstats is not None else None,
            "final_candidates": _safe_int(fstats.final_total) if fstats is not None else None,
            "removed_abs": _safe_int(fstats.removed_abs) if fstats is not None else None,
            "removed_pct": _safe_float(fstats.removed_pct) if fstats is not None else None,
            **res_summary,
        }

    except Exception as e:
        
        partial_res_summary = {}
        if res is not None:
            try:
                partial_res_summary = {f"{k}": v for k, v in res.summary().items()}
            except Exception:
                partial_res_summary = {}
        
        return {
            "label": label,
            "duration_hours": duration_hours,
            "n_queries": n_queries,
            "repeat_idx": repeat_idx,
            "seed": seed,
            "success": False,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "ci_cov": _safe_float(cp.ci_cov()) if cp is not None else None,
            "num_slots": _safe_int(cp.num_slots) if cp is not None else None,
            "num_queries_actual": _safe_int(len(specs_true)) if specs_true is not None else None,
            "raw_candidates": _safe_int(fstats.raw_total) if fstats is not None else None,
            "final_candidates": _safe_int(fstats.final_total) if fstats is not None else None,
            "removed_abs": _safe_int(fstats.removed_abs) if fstats is not None else None,
            "removed_pct": _safe_float(fstats.removed_pct) if fstats is not None else None,
            **partial_res_summary,
        }


# duration -> label -> n_queries -> repeats
def evaluate_scaling_grid(
    *,
    csv_path: str,
    start: str,
    durations_hours: list[int],
    query_counts: list[int],
    n_repeats: int = 5,
    penalty_sets: Optional[dict[str, list]] = None,
    seed: int = 42,
    reopt_threshold: float = 0.5,
    verbose_logger: bool = False,
    verbose_filter: bool = False,
    out_csv: Path = Path("./results.csv").resolve(),
) -> pd.DataFrame:
    if penalty_sets is None:
        penalty_sets = {"no_policy": []}

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    rows = []

    for duration_hours in durations_hours:
        for label, penalties in penalty_sets.items():
            prune_larger_query_counts = False
    
            for n_queries in query_counts:
                if prune_larger_query_counts:
                    break
    
                for repeat_idx in range(n_repeats):
                    print(
                        f"Running: label={label}, duration={duration_hours}h, "
                        f"queries={n_queries}, repeat={repeat_idx+1}/{n_repeats}, seed={seed}"
                    )
    
                    row = run_single_scheduler_trial(
                        csv_path=csv_path,
                        start=start,
                        duration_hours=duration_hours,
                        n_queries=n_queries,
                        label=label,
                        penalties=penalties,
                        repeat_idx=repeat_idx,
                        seed=seed,
                        reopt_threshold=reopt_threshold,
                        verbose_logger=verbose_logger,
                        verbose_filter=verbose_filter,
                    )
    
                    pd.DataFrame([row]).to_csv(
                        out_csv,
                        mode="a",
                        header=not out_csv.exists(),
                        index=False,
                    )
                    rows.append(row)
    
                    if row["success"] is False:
                        prune_larger_query_counts = True
                        break

    return pd.DataFrame(rows)


def aggregate_scaling_results(df_trials: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate over successful runs only, but keep counts of failures.
    """
    group_cols = ["label", "duration_hours", "n_queries"]

    # Failure / success summary
    counts = (
        df_trials.groupby(group_cols, dropna=False)
        .agg(
            n_runs=("success", "size"),
            n_success=("success", "sum"),
        )
        .reset_index()
    )
    counts["n_failed"] = counts["n_runs"] - counts["n_success"]
    counts["success_rate"] = counts["n_success"] / counts["n_runs"]

    # Numeric averages over successful runs only
    df_ok = df_trials[df_trials["success"]].copy()

    metric_cols = [
        "ci_cov",
        "num_slots",
        "num_queries_actual",
        "raw_candidates",
        "final_candidates",
        "removed_abs",
        "removed_pct",
        "wall_seconds",
        "wall_seconds",
        "num_reopts",
        "num_idle_jumps",
        "total_idle_slots",
        "finish_slot",
        "planned_carbon",
        "realised_carbon",
    ]

    agg_dict = {}
    for c in metric_cols:
        agg_dict[f"{c}_mean"] = (c, "mean")
        agg_dict[f"{c}_std"] = (c, "std")
        agg_dict[f"{c}_min"] = (c, "min")
        agg_dict[f"{c}_max"] = (c, "max")

    summary = (
        df_ok.groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )

    out = counts.merge(summary, on=group_cols, how="left")
    return out

In [2]:
durations_hours = [12]   # 12h to 1 week
query_counts = [50, 100, 150, 200, 250, 300]


penalty_sets = {
    "objrisk_window_ci": [WindowCIRiskPenalty(lam=0.5, power=1.0, weight=1)],
}

df_trials = evaluate_scaling_grid(
    csv_path=CSV_PATH,
    start=START,
    durations_hours=durations_hours,
    query_counts=query_counts,
    n_repeats=1,
    penalty_sets=penalty_sets,
    seed=SEED,
    reopt_threshold=0.5,
    verbose_logger=False,
    verbose_filter=False,
    out_csv=Path("./Studies/Efficenciey/grid_new.csv").resolve(),
)

df_summary = aggregate_scaling_results(df_trials)

Running: label=objrisk_window_ci, duration=12h, queries=50, repeat=1/1, seed=42
Running: label=objrisk_window_ci, duration=12h, queries=100, repeat=1/1, seed=42
Running: label=objrisk_window_ci, duration=12h, queries=150, repeat=1/1, seed=42
Running: label=objrisk_window_ci, duration=12h, queries=200, repeat=1/1, seed=42
Running: label=objrisk_window_ci, duration=12h, queries=250, repeat=1/1, seed=42


In [10]:
import matplotlib.pyplot as plt

query_counts = [0, 50, 100, 150, 200, 250, 300]
totals_hours = []

for nq in query_counts:
    if nq == 0:
        total_runtime_hours = 0.0
    else:
        df_test = synthesize_df_test(
            n_queries=nq,
            n_runs=N_RUNS,
            base_median=BASE_MEDIAN,
            max_median=MAX_MEDIAN,
            cpu_min=CPU_MIN,
            cpu_max=CPU_MAX,
            ram_min=RAM_MIN,
            ram_max=RAM_MAX,
            priority_high=PRIORITY_HIGH,
            priority_low=PRIORITY_LOW,
            priority_w0=PRIORITY_W0,
            priority_w1=PRIORITY_w1,
            seed=42,
        )

        total_runtime_hours = df_test["runtime_mean_ac"].sum() / 3600.0

    totals_hours.append(total_runtime_hours)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(x) for x in query_counts], totals_hours)

ax.set_xlabel("Number of queries")
ax.set_ylabel("Total mean predicted runtime (hours)")
ax.set_title("Total predicted runtime by workload size")

for i, v in enumerate(totals_hours):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
plt.show()

KeyError: 'runtime_mean_ac'